# HIV-Hi — OOD Holdout vs Random Shuffle Protocol Comparison

This notebook compares two inner hyperparameter-selection protocols on the HIV-Hi task:

1. **OOD holdout**: the validation subset is chemically separated from the inner training subset.
2. **Random shuffle**: the validation subset is obtained by randomly splitting the outer training set.

The main experimental question is:

> Does in-distribution hyperparameter selection produce higher internal validation scores without improving final out-of-distribution test performance?

The analysis focuses on:

- inner validation PR-AUC;
- inner train PR-AUC;
- outer train PR-AUC after refitting;
- final OOD test PR-AUC;
- inner-to-test generalization gap;
- train-to-test generalization gap;
- selected hyperparameters across folds.

The models considered are:

- Decision Tree;
- Logistic Regression;
- Linear SVM.

The fingerprints considered are:

- ECFP4;
- MACCS;
- RDKit descriptors, where available.

In [19]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path("../..").resolve()
RESULTS_ROOT = PROJECT_ROOT / "results" / "hi" / "hiv"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RESULTS_ROOT: {RESULTS_ROOT}")
print(f"RESULTS_ROOT exists: {RESULTS_ROOT.exists()}")

PROJECT_ROOT: /home/f.capria/drug-discovery-lohi
RESULTS_ROOT: /home/f.capria/drug-discovery-lohi/results/hi/hiv
RESULTS_ROOT exists: True


# Experiment registry

In [20]:
EXPERIMENTS = [
    # ---------------------------------------------------------------------
    # Decision Tree
    # ---------------------------------------------------------------------
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "dt_hiv_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "dt_hiv_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "dt_hiv_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Decision Tree",
        "model_short": "DT",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "dt_hiv_hi_random_shuffle_maccs",
    },

    # ---------------------------------------------------------------------
    # Logistic Regression
    # ---------------------------------------------------------------------
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "lr_hiv_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "lr_hiv_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "OOD holdout",
        "result_dir": "lr_hiv_hi_inner_ood_holdout_rdkit_desc",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "lr_hiv_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "lr_hiv_hi_random_shuffle_maccs",
    },
    {
        "model": "Logistic Regression",
        "model_short": "LR",
        "fingerprint": "RDKit desc",
        "protocol": "Random shuffle",
        "result_dir": "lr_hiv_hi_random_shuffle_rdkit_desc",
    },

    # ---------------------------------------------------------------------
    # Linear SVM
    # ---------------------------------------------------------------------
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_hiv_hi_inner_ood_holdout_ecfp4",
    },
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "OOD holdout",
        "result_dir": "svm_linear_hiv_hi_inner_ood_holdout_maccs",
    },
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "ECFP4",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_hiv_hi_random_shuffle_ecfp4",
    },
    {
        "model": "Linear SVM",
        "model_short": "SVM",
        "fingerprint": "MACCS",
        "protocol": "Random shuffle",
        "result_dir": "svm_linear_hiv_hi_random_shuffle_maccs",
    },
]

registry = pd.DataFrame(EXPERIMENTS)
registry["path"] = registry["result_dir"].apply(lambda d: RESULTS_ROOT / d)
registry["exists"] = registry["path"].apply(lambda p: p.exists())

registry

,model,model_short,fingerprint,protocol,result_dir,path,exists
0,Decision Tree,DT,ECFP4,OOD holdout,dt_hiv_hi_inner_ood_holdout_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/hiv/dt_hiv_hi_inner_ood_holdout_ecfp4,True
1,Decision Tree,DT,MACCS,OOD holdout,dt_hiv_hi_inner_ood_holdout_maccs,/home/f.capria/drug-discovery-lohi/results/hi/hiv/dt_hiv_hi_inner_ood_holdout_maccs,True
2,Decision Tree,DT,ECFP4,Random shuffle,dt_hiv_hi_random_shuffle_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/hiv/dt_hiv_hi_random_shuffle_ecfp4,True
3,Decision Tree,DT,MACCS,Random shuffle,dt_hiv_hi_random_shuffle_maccs,/home/f.capria/drug-discovery-lohi/results/hi/hiv/dt_hiv_hi_random_shuffle_maccs,True
4,Logistic Regression,LR,ECFP4,OOD holdout,lr_hiv_hi_inner_ood_holdout_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/hiv/lr_hiv_hi_inner_ood_holdout_ecfp4,True
5,Logistic Regression,LR,MACCS,OOD holdout,lr_hiv_hi_inner_ood_holdout_maccs,/home/f.capria/drug-discovery-lohi/results/hi/hiv/lr_hiv_hi_inner_ood_holdout_maccs,True
6,Logistic Regression,LR,RDKit desc,OOD holdout,lr_hiv_hi_inner_ood_holdout_rdkit_desc,/home/f.capria/drug-discovery-lohi/results/hi/hiv/lr_hiv_hi_inner_ood_holdout_rdkit_desc,True
7,Logistic Regression,LR,ECFP4,Random shuffle,lr_hiv_hi_random_shuffle_ecfp4,/home/f.capria/drug-discovery-lohi/results/hi/hiv/lr_hiv_hi_random_shuffle_ecfp4,True
8,Logistic Regression,LR,MACCS,Random shuffle,lr_hiv_hi_random_shuffle_maccs,/home/f.capria/drug-discovery-lohi/results/hi/hiv/lr_hiv_hi_random_shuffle_maccs,True
9,Logistic Regression,LR,RDKit desc,Random shuffle,lr_hiv_hi_random_shuffle_rdkit_desc,/home/f.capria/drug-discovery-lohi/results/hi/hiv/lr_hiv_hi_random_shuffle_rdkit_desc,True


In [21]:
missing = registry.loc[~registry["exists"], ["model", "fingerprint", "protocol", "path"]]

if len(missing) > 0:
    print("Missing result folders:")
    display(missing)
else:
    print("All registered result folders exist.")

All registered result folders exist.


# Load params_fold_i.json into df_folds

In [22]:
def load_params_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)


def safe_get_metric(metrics: dict, key: str):
    if metrics is None:
        return np.nan
    return metrics.get(key, np.nan)


rows = []

for exp in EXPERIMENTS:
    result_path = RESULTS_ROOT / exp["result_dir"]

    for fold in [1, 2, 3]:
        params_path = result_path / f"params_fold_{fold}.json"

        if not params_path.exists():
            print(f"Missing file: {params_path}")
            continue

        data = load_params_json(params_path)

        train_metrics = data.get("train_metrics", {})
        test_metrics = data.get("test_metrics", {})
        best_params = data.get("best_params", {})

        inner_pr_auc = data.get("inner_selection_score", np.nan)
        inner_train_pr_auc = data.get("inner_train_score", np.nan)
        train_pr_auc = safe_get_metric(train_metrics, "pr_auc")
        test_pr_auc = safe_get_metric(test_metrics, "pr_auc")

        row = {
            "model": exp["model"],
            "model_short": exp["model_short"],
            "fingerprint": exp["fingerprint"],
            "protocol": exp["protocol"],
            "result_dir": exp["result_dir"],
            "fold": fold,

            "inner_pr_auc": inner_pr_auc,
            "inner_train_pr_auc": inner_train_pr_auc,
            "train_pr_auc": train_pr_auc,
            "test_pr_auc": test_pr_auc,

            "inner_test_gap": inner_pr_auc - test_pr_auc,
            "train_test_gap": train_pr_auc - test_pr_auc,

            "best_params": best_params,
            "inner_split_strategy": data.get("inner_split_strategy", None),
            "time_seconds": data.get("time_seconds", np.nan),
        }

        rows.append(row)

df_folds = pd.DataFrame(rows)

order_model = {
    "Decision Tree": 0,
    "Logistic Regression": 1,
    "Linear SVM": 2,
}

order_fp = {
    "ECFP4": 0,
    "MACCS": 1,
    "RDKit desc": 2,
}

order_protocol = {
    "OOD holdout": 0,
    "Random shuffle": 1,
}

df_folds["model_order"] = df_folds["model"].map(order_model)
df_folds["fingerprint_order"] = df_folds["fingerprint"].map(order_fp)
df_folds["protocol_order"] = df_folds["protocol"].map(order_protocol)

df_folds = df_folds.sort_values(
    ["model_order", "fingerprint_order", "protocol_order", "fold"]
).reset_index(drop=True)

print(f"Loaded rows: {len(df_folds)}")
df_folds.head(2)

Loaded rows: 42


,model,model_short,fingerprint,protocol,result_dir,fold,inner_pr_auc,inner_train_pr_auc,train_pr_auc,test_pr_auc,inner_test_gap,train_test_gap,best_params,inner_split_strategy,time_seconds,model_order,fingerprint_order,protocol_order
0,Decision Tree,DT,ECFP4,OOD holdout,dt_hiv_hi_inner_ood_holdout_ecfp4,1,0.112536,0.032367,0.1173,0.0395,0.073036,0.0778,"{'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'entropy', 'max_depth': 3, 'max_features': 'log2', 'min_sample...",holdout,9.1,0,0,0
1,Decision Tree,DT,ECFP4,OOD holdout,dt_hiv_hi_inner_ood_holdout_ecfp4,2,0.087381,0.097746,0.2730,0.0748,0.012581,0.1982,"{'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 15, 'max_features': 'log2', 'min_samples_...",holdout,6.4,0,0,0


# Per-fold table

In [23]:
per_fold_table = df_folds[
    [
        "model",
        "fingerprint",
        "protocol",
        "fold",
        "inner_pr_auc",
        "inner_train_pr_auc",
        "train_pr_auc",
        "test_pr_auc",
        "inner_test_gap",
        "train_test_gap",
    ]
].copy()

numeric_cols = [
    "inner_pr_auc",
    "inner_train_pr_auc",
    "train_pr_auc",                                                                                         
    "test_pr_auc",
    "inner_test_gap",
    "train_test_gap",
]

per_fold_table[numeric_cols] = per_fold_table[numeric_cols].round(4)

per_fold_table

,model,fingerprint,protocol,fold,inner_pr_auc,inner_train_pr_auc,train_pr_auc,test_pr_auc,inner_test_gap,train_test_gap
0,Decision Tree,ECFP4,OOD holdout,1,0.1125,0.0324,0.1173,0.0395,0.0730,0.0778
1,Decision Tree,ECFP4,OOD holdout,2,0.0874,0.0977,0.2730,0.0748,0.0126,0.1982
2,Decision Tree,ECFP4,OOD holdout,3,0.0830,0.5457,0.5066,0.0257,0.0573,0.4809
3,Decision Tree,ECFP4,Random shuffle,1,0.3496,0.4077,0.4334,0.0893,0.2603,0.3441
4,Decision Tree,ECFP4,Random shuffle,2,0.3592,0.4102,0.3868,0.0616,0.2976,0.3252
5,Decision Tree,ECFP4,Random shuffle,3,0.3936,0.5137,0.5079,0.0279,0.3657,0.4800
6,Decision Tree,MACCS,OOD holdout,1,0.1218,0.0933,0.2458,0.1470,-0.0252,0.0988
7,Decision Tree,MACCS,OOD holdout,2,0.2083,0.1630,0.3699,0.0613,0.1470,0.3086
8,Decision Tree,MACCS,OOD holdout,3,0.2082,0.4185,0.4180,0.0250,0.1832,0.3930
9,Decision Tree,MACCS,Random shuffle,1,0.3328,0.4991,0.4744,0.0584,0.2744,0.4160


# Aggregated summary table

In [24]:
def mean_std_string(mean_value, std_value, digits=4):
    if pd.isna(mean_value):
        return ""
    if pd.isna(std_value):
        return f"{mean_value:.{digits}f}"
    return f"{mean_value:.{digits}f} ± {std_value:.{digits}f}"


summary_numeric = (
    df_folds
    .groupby(["model", "model_short", "fingerprint", "protocol"], as_index=False)
    .agg(
        inner_mean=("inner_pr_auc", "mean"),
        inner_std=("inner_pr_auc", "std"),
        inner_train_mean=("inner_train_pr_auc", "mean"),
        inner_train_std=("inner_train_pr_auc", "std"),
        train_mean=("train_pr_auc", "mean"),
        train_std=("train_pr_auc", "std"),
        test_mean=("test_pr_auc", "mean"),
        test_std=("test_pr_auc", "std"),
        inner_test_gap_mean=("inner_test_gap", "mean"),
        inner_test_gap_std=("inner_test_gap", "std"),
        train_test_gap_mean=("train_test_gap", "mean"),
        train_test_gap_std=("train_test_gap", "std"),
    )
)

summary_numeric["model_order"] = summary_numeric["model"].map(order_model)
summary_numeric["fingerprint_order"] = summary_numeric["fingerprint"].map(order_fp)
summary_numeric["protocol_order"] = summary_numeric["protocol"].map(order_protocol)

summary_numeric = summary_numeric.sort_values(
    ["model_order", "fingerprint_order", "protocol_order"]
).reset_index(drop=True)

summary_table = summary_numeric[
    ["model", "fingerprint", "protocol"]
].copy()

summary_table["Inner PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["inner_mean"], r["inner_std"]), axis=1
)

summary_table["Inner train PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["inner_train_mean"], r["inner_train_std"]), axis=1
)

summary_table["Train PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["train_mean"], r["train_std"]), axis=1
)

summary_table["Final OOD test PR-AUC"] = summary_numeric.apply(
    lambda r: mean_std_string(r["test_mean"], r["test_std"]), axis=1
)

summary_table["Inner-test gap"] = summary_numeric.apply(
    lambda r: mean_std_string(r["inner_test_gap_mean"], r["inner_test_gap_std"]), axis=1
)

summary_table["Train-test gap"] = summary_numeric.apply(
    lambda r: mean_std_string(r["train_test_gap_mean"], r["train_test_gap_std"]), axis=1
)

summary_table

,model,fingerprint,protocol,Inner PR-AUC,Inner train PR-AUC,Train PR-AUC,Final OOD test PR-AUC,Inner-test gap,Train-test gap
0,Decision Tree,ECFP4,OOD holdout,0.0943 ± 0.0159,0.2253 ± 0.2794,0.2990 ± 0.1959,0.0467 ± 0.0253,0.0477 ± 0.0314,0.2523 ± 0.2069
1,Decision Tree,ECFP4,Random shuffle,0.3675 ± 0.0231,0.4439 ± 0.0605,0.4427 ± 0.0611,0.0596 ± 0.0307,0.3079 ± 0.0534,0.3831 ± 0.0844
2,Decision Tree,MACCS,OOD holdout,0.1794 ± 0.0499,0.2249 ± 0.1712,0.3446 ± 0.0889,0.0778 ± 0.0626,0.1017 ± 0.1113,0.2668 ± 0.1515
3,Decision Tree,MACCS,Random shuffle,0.3893 ± 0.0490,0.5141 ± 0.0676,0.5079 ± 0.0655,0.0536 ± 0.0148,0.3356 ± 0.0540,0.4543 ± 0.0801
4,Logistic Regression,ECFP4,OOD holdout,0.1285 ± 0.0328,0.3705 ± 0.2862,0.4534 ± 0.0661,0.0577 ± 0.0241,0.0708 ± 0.0558,0.3958 ± 0.0893
5,Logistic Regression,ECFP4,Random shuffle,0.4753 ± 0.0447,0.6042 ± 0.0594,0.5997 ± 0.0594,0.0714 ± 0.0375,0.4039 ± 0.0822,0.5284 ± 0.0944
6,Logistic Regression,MACCS,OOD holdout,0.1753 ± 0.0637,0.1619 ± 0.1368,0.2281 ± 0.0713,0.1129 ± 0.0775,0.0624 ± 0.1376,0.1153 ± 0.1223
7,Logistic Regression,MACCS,Random shuffle,0.3424 ± 0.0425,0.3581 ± 0.0357,0.3648 ± 0.0358,0.1001 ± 0.0594,0.2422 ± 0.1008,0.2647 ± 0.0892
8,Logistic Regression,RDKit desc,OOD holdout,0.1721 ± 0.0476,0.3035 ± 0.2219,0.4238 ± 0.0842,0.0885 ± 0.0473,0.0836 ± 0.0221,0.3353 ± 0.1152
9,Logistic Regression,RDKit desc,Random shuffle,0.3779 ± 0.0661,0.4171 ± 0.0758,0.4176 ± 0.0693,0.0887 ± 0.0453,0.2891 ± 0.0848,0.3289 ± 0.1091


# Delta table

In [25]:
pivot_summary = summary_numeric.pivot_table(
    index=["model", "model_short", "fingerprint"],
    columns="protocol",
    values=[
        "inner_mean",
        "test_mean",
        "inner_test_gap_mean",
        "train_test_gap_mean",
    ],
)

delta_rows = []

for idx, row in pivot_summary.iterrows():
    model, model_short, fingerprint = idx

    try:
        random_inner = row[("inner_mean", "Random shuffle")]
        ood_inner = row[("inner_mean", "OOD holdout")]

        random_test = row[("test_mean", "Random shuffle")]
        ood_test = row[("test_mean", "OOD holdout")]

        random_inner_gap = row[("inner_test_gap_mean", "Random shuffle")]
        ood_inner_gap = row[("inner_test_gap_mean", "OOD holdout")]

        random_train_gap = row[("train_test_gap_mean", "Random shuffle")]
        ood_train_gap = row[("train_test_gap_mean", "OOD holdout")]

    except KeyError:
        continue

    delta_rows.append({
        "model": model,
        "model_short": model_short,
        "fingerprint": fingerprint,

        "ood_inner_mean": ood_inner,
        "random_inner_mean": random_inner,
        "delta_inner": random_inner - ood_inner,

        "ood_test_mean": ood_test,
        "random_test_mean": random_test,
        "delta_test": random_test - ood_test,

        "ood_inner_test_gap": ood_inner_gap,
        "random_inner_test_gap": random_inner_gap,
        "delta_inner_test_gap": random_inner_gap - ood_inner_gap,

        "ood_train_test_gap": ood_train_gap,
        "random_train_test_gap": random_train_gap,
        "delta_train_test_gap": random_train_gap - ood_train_gap,
    })

delta_table = pd.DataFrame(delta_rows)

delta_table["model_order"] = delta_table["model"].map(order_model)
delta_table["fingerprint_order"] = delta_table["fingerprint"].map(order_fp)

delta_table = delta_table.sort_values(
    ["model_order", "fingerprint_order"]
).reset_index(drop=True)

delta_display = delta_table[
    [
        "model",
        "fingerprint",
        "ood_inner_mean",
        "random_inner_mean",
        "delta_inner",
        "ood_test_mean",
        "random_test_mean",
        "delta_test",
        "ood_inner_test_gap",
        "random_inner_test_gap",
        "delta_inner_test_gap",
        "delta_train_test_gap",
    ]
].copy()

delta_numeric_cols = delta_display.select_dtypes(include=[np.number]).columns
delta_display[delta_numeric_cols] = delta_display[delta_numeric_cols].round(4)

delta_display

,model,fingerprint,ood_inner_mean,random_inner_mean,delta_inner,ood_test_mean,random_test_mean,delta_test,ood_inner_test_gap,random_inner_test_gap,delta_inner_test_gap,delta_train_test_gap
0,Decision Tree,ECFP4,0.0943,0.3675,0.2731,0.0467,0.0596,0.0129,0.0477,0.3079,0.2602,0.1308
1,Decision Tree,MACCS,0.1794,0.3893,0.2098,0.0778,0.0536,-0.0241,0.1017,0.3356,0.2340,0.1875
2,Logistic Regression,ECFP4,0.1285,0.4753,0.3468,0.0577,0.0714,0.0137,0.0708,0.4039,0.3331,0.1326
3,Logistic Regression,MACCS,0.1753,0.3424,0.1671,0.1129,0.1001,-0.0127,0.0624,0.2422,0.1798,0.1494
4,Logistic Regression,RDKit desc,0.1721,0.3779,0.2058,0.0885,0.0887,0.0002,0.0836,0.2891,0.2056,-0.0065
5,Linear SVM,ECFP4,0.0982,0.4333,0.3351,0.0532,0.0728,0.0196,0.0450,0.3605,0.3155,-0.0502
6,Linear SVM,MACCS,0.1630,0.2816,0.1186,0.1034,0.1061,0.0027,0.0595,0.1755,0.1159,0.0126


# Hyperparameter summary tables

In [26]:
def flatten_best_params(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in df.iterrows():
        base = {
            "model": row["model"],
            "fingerprint": row["fingerprint"],
            "protocol": row["protocol"],
            "fold": row["fold"],
            "inner_pr_auc": row["inner_pr_auc"],
            "test_pr_auc": row["test_pr_auc"],
        }

        params = row["best_params"]

        if isinstance(params, dict):
            for key, value in params.items():
                clean_key = key.replace("model__", "")
                base[clean_key] = value

        rows.append(base)

    return pd.DataFrame(rows)


hp_df = flatten_best_params(df_folds)

hp_df["model_order"] = hp_df["model"].map(order_model)
hp_df["fingerprint_order"] = hp_df["fingerprint"].map(order_fp)
hp_df["protocol_order"] = hp_df["protocol"].map(order_protocol)

hp_df = hp_df.sort_values(
    ["model_order", "fingerprint_order", "protocol_order", "fold"]
).reset_index(drop=True)

for col in ["inner_pr_auc", "test_pr_auc"]:
    hp_df[col] = hp_df[col].round(4)

hp_df.head()

,model,fingerprint,protocol,fold,inner_pr_auc,test_pr_auc,ccp_alpha,class_weight,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,C,l1_ratio,model_order,fingerprint_order,protocol_order
0,Decision Tree,ECFP4,OOD holdout,1,0.1125,0.0395,0.0000,NaN,entropy,3.0,log2,5.0,10.0,NaN,NaN,0,0,0
1,Decision Tree,ECFP4,OOD holdout,2,0.0874,0.0748,0.0000,NaN,gini,15.0,log2,5.0,20.0,NaN,NaN,0,0,0
2,Decision Tree,ECFP4,OOD holdout,3,0.0830,0.0257,0.0001,balanced,entropy,15.0,sqrt,5.0,10.0,NaN,NaN,0,0,0
3,Decision Tree,ECFP4,Random shuffle,1,0.3496,0.0893,0.0000,NaN,gini,15.0,sqrt,5.0,20.0,NaN,NaN,0,0,1
4,Decision Tree,ECFP4,Random shuffle,2,0.3592,0.0616,0.0000,balanced,gini,15.0,sqrt,5.0,20.0,NaN,NaN,0,0,1


# Logistic Regression hyperparameters

In [27]:
lr_hp_table = hp_df.loc[
    hp_df["model"] == "Logistic Regression",
    [
        "protocol",
        "fingerprint",
        "fold",
        "C",
        "class_weight",
        "l1_ratio",
        "inner_pr_auc",
        "test_pr_auc",
    ]
].copy()

lr_hp_table = lr_hp_table.sort_values(
    ["fingerprint", "protocol", "fold"]
).reset_index(drop=True)

lr_hp_table

,protocol,fingerprint,fold,C,class_weight,l1_ratio,inner_pr_auc,test_pr_auc
0,OOD holdout,ECFP4,1,0.10,NaN,1.0,0.1025,0.0676
1,OOD holdout,ECFP4,2,0.10,balanced,0.0,0.1176,0.0752
2,OOD holdout,ECFP4,3,0.10,balanced,1.0,0.1653,0.0302
3,Random shuffle,ECFP4,1,0.10,NaN,0.0,0.4508,0.0906
4,Random shuffle,ECFP4,2,0.10,NaN,0.0,0.4482,0.0954
5,Random shuffle,ECFP4,3,0.10,NaN,0.0,0.5269,0.0281
6,OOD holdout,MACCS,1,10.00,balanced,1.0,0.1217,0.1984
7,OOD holdout,MACCS,2,0.10,NaN,1.0,0.1585,0.0929
8,OOD holdout,MACCS,3,0.10,balanced,1.0,0.2457,0.0473
9,Random shuffle,MACCS,1,5.00,NaN,0.5,0.3048,0.1634


# Svm liner hyperparameters

In [28]:
svm_hp_table = hp_df.loc[
    hp_df["model"] == "Linear SVM",
    [
        "protocol",
        "fingerprint",
        "fold",
        "C",
        "class_weight",
        "inner_pr_auc",
        "test_pr_auc",
    ]
].copy()

svm_hp_table = svm_hp_table.sort_values(
    ["fingerprint", "protocol", "fold"]
).reset_index(drop=True)

svm_hp_table

,protocol,fingerprint,fold,C,class_weight,inner_pr_auc,test_pr_auc
0,OOD holdout,ECFP4,1,0.10,NaN,0.0779,0.0616
1,OOD holdout,ECFP4,2,0.50,NaN,0.1111,0.0675
2,OOD holdout,ECFP4,3,0.01,balanced,0.1057,0.0306
3,Random shuffle,ECFP4,1,0.01,balanced,0.3970,0.1223
4,Random shuffle,ECFP4,2,0.10,NaN,0.3951,0.0721
5,Random shuffle,ECFP4,3,0.10,NaN,0.5078,0.0241
6,OOD holdout,MACCS,1,0.10,balanced,0.0999,0.1456
7,OOD holdout,MACCS,2,0.10,NaN,0.2141,0.1210
8,OOD holdout,MACCS,3,0.01,balanced,0.1748,0.0437
9,Random shuffle,MACCS,1,0.01,NaN,0.2160,0.1738


# Decision Tree hyperparameters

In [29]:
dt_hp_table = hp_df.loc[
    hp_df["model"] == "Decision Tree",
    [
        "protocol",
        "fingerprint",
        "fold",
        "criterion",
        "max_depth",
        "max_features",
        "min_samples_leaf",
        "min_samples_split",
        "ccp_alpha",
        "class_weight",
        "inner_pr_auc",
        "test_pr_auc",
    ]
].copy()

dt_hp_table = dt_hp_table.sort_values(
    ["fingerprint", "protocol", "fold"]
).reset_index(drop=True)

dt_hp_table

,protocol,fingerprint,fold,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,ccp_alpha,class_weight,inner_pr_auc,test_pr_auc
0,OOD holdout,ECFP4,1,entropy,3.0,log2,5.0,10.0,0.0000,NaN,0.1125,0.0395
1,OOD holdout,ECFP4,2,gini,15.0,log2,5.0,20.0,0.0000,NaN,0.0874,0.0748
2,OOD holdout,ECFP4,3,entropy,15.0,sqrt,5.0,10.0,0.0001,balanced,0.0830,0.0257
3,Random shuffle,ECFP4,1,gini,15.0,sqrt,5.0,20.0,0.0000,NaN,0.3496,0.0893
4,Random shuffle,ECFP4,2,gini,15.0,sqrt,5.0,20.0,0.0000,balanced,0.3592,0.0616
5,Random shuffle,ECFP4,3,gini,15.0,sqrt,5.0,10.0,0.0000,balanced,0.3936,0.0279
6,OOD holdout,MACCS,1,entropy,8.0,log2,5.0,50.0,0.0001,NaN,0.1218,0.1470
7,OOD holdout,MACCS,2,entropy,15.0,sqrt,20.0,10.0,0.0000,NaN,0.2083,0.0613
8,OOD holdout,MACCS,3,gini,10.0,sqrt,10.0,50.0,0.0000,balanced,0.2082,0.0250
9,Random shuffle,MACCS,1,gini,15.0,sqrt,5.0,20.0,0.0000,NaN,0.3328,0.0584


# Compact hyperparameter set summary

In [30]:
def unique_values_as_string(series: pd.Series) -> str:
    values = []
    for value in series.dropna().tolist():
        if value not in values:
            values.append(value)
    return "{" + ", ".join(str(v) for v in values) + "}"


hp_set_summary_rows = []

for model_name, model_df in hp_df.groupby("model"):
    param_cols = [
        c for c in model_df.columns
        if c not in [
            "model",
            "fingerprint",
            "protocol",
            "fold",
            "inner_pr_auc",
            "test_pr_auc",
            "model_order",
            "fingerprint_order",
            "protocol_order",
        ]
    ]

    for protocol, protocol_df in model_df.groupby("protocol"):
        row = {
            "model": model_name,
            "protocol": protocol,
        }

        for col in param_cols:
            if protocol_df[col].notna().any():
                row[col] = unique_values_as_string(protocol_df[col])

        hp_set_summary_rows.append(row)

hp_set_summary = pd.DataFrame(hp_set_summary_rows)
hp_set_summary["model_order"] = hp_set_summary["model"].map(order_model)
hp_set_summary["protocol_order"] = hp_set_summary["protocol"].map(order_protocol)

hp_set_summary = hp_set_summary.sort_values(
    ["model_order", "protocol_order"]
).drop(columns=["model_order", "protocol_order"]).reset_index(drop=True)

hp_set_summary

,model,protocol,ccp_alpha,class_weight,criterion,max_depth,max_features,min_samples_leaf,min_samples_split,C,l1_ratio
0,Decision Tree,OOD holdout,"{0.0, 0.0001}",{balanced},"{entropy, gini}","{3.0, 15.0, 8.0, 10.0}","{log2, sqrt}","{5.0, 20.0, 10.0}","{10.0, 20.0, 50.0}",NaN,NaN
1,Decision Tree,Random shuffle,"{0.0, 0.0001}",{balanced},"{gini, entropy}",{15.0},{sqrt},"{5.0, 10.0}","{20.0, 10.0}",NaN,NaN
2,Logistic Regression,OOD holdout,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{0.1, 10.0, 0.01}","{1.0, 0.0}"
3,Logistic Regression,Random shuffle,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{0.1, 5.0, 1.0}","{0.0, 0.5, 1.0}"
4,Linear SVM,OOD holdout,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{0.1, 0.5, 0.01}",NaN
5,Linear SVM,Random shuffle,NaN,{balanced},NaN,NaN,NaN,NaN,NaN,"{0.01, 0.1, 5.0, 0.5}",NaN


# Save processed tables for the plotting notebook

In [31]:
TASK = "hi"
DATASET = "hiv"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "results_ood_vs_random_shuffle"
    / TASK
    / DATASET
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Main protocol comparison tables
df_folds.to_csv(
    OUTPUT_DIR / "protocol_per_fold.csv",
    index=False,
)

summary_numeric.to_csv(
    OUTPUT_DIR / "protocol_summary_numeric.csv",
    index=False,
)

summary_table.to_csv(
    OUTPUT_DIR / "protocol_summary_display.csv",
    index=False,
)

delta_table.to_csv(
    OUTPUT_DIR / "protocol_delta.csv",
    index=False,
)

# Hyperparameter tables
hp_df.to_csv(
    OUTPUT_DIR / "hyperparameters_all.csv",
    index=False,
)

lr_hp_table.to_csv(
    OUTPUT_DIR / "hyperparameters_lr.csv",
    index=False,
)

svm_hp_table.to_csv(
    OUTPUT_DIR / "hyperparameters_svm.csv",
    index=False,
)

dt_hp_table.to_csv(
    OUTPUT_DIR / "hyperparameters_dt.csv",
    index=False,
)

hp_set_summary.to_csv(
    OUTPUT_DIR / "hyperparameters_set_summary.csv",
    index=False,
)

print("Saved processed files in:")
print(OUTPUT_DIR)

print("\nFiles saved:")
for file in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"- {file.name}")

Saved processed files in:
/home/f.capria/drug-discovery-lohi/results/results_ood_vs_random_shuffle/hi/hiv

Files saved:
- hyperparameters_all.csv
- hyperparameters_dt.csv
- hyperparameters_lr.csv
- hyperparameters_set_summary.csv
- hyperparameters_svm.csv
- protocol_delta.csv
- protocol_per_fold.csv
- protocol_summary_display.csv
- protocol_summary_numeric.csv
